# box-array-to-tensor-with-recipe composite — cx10: round-trip array ↔ MiniTensor with shared storage (no copy)

> Composite procedural drill from [Delta Drills](https://delta-drills.vercel.app).
> Exercises 2 atoms together: `box-array-to-tensor-with-recipe`, `unbox-args-tensor-to-array`
> Running the final beacon reports progress against all 2 subtopics.

**Why composite drills.** Single-atom drills test atomic skills in isolation. Composite drills test the COMPOSITION — how atoms wire together in real ARENA code. Passing this drill demonstrates you can apply the atoms jointly, not just individually.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)

## Connect to Delta Drills

Paste your Delta Drills auth token below. Beacon will report progress against ALL atoms exercised by this composite.

In [ ]:
# === Delta Drills auth (composite) ===
DD_TOKEN = ""  # paste token, then run
DD_PRIMARY_ATOM = "box-array-to-tensor-with-recipe"
DD_ATOM_IDS = ["box-array-to-tensor-with-recipe", "unbox-args-tensor-to-array"]
DD_SUBTOPICS = ["Backprop: Box array as Tensor + recipe", "Backprop: Unbox Tensor args to array"]
DD_BACKEND_URL = "https://delta-drills-backend.fly.dev"

_dd_passed = set()

## Round-tripping arrays through MiniTensor — identity, not copy

Both halves of `wrap_forward_fn` are deliberately **zero-copy**:

- **Unbox**: replace each `MiniTensor` with its `.array` — the same underlying raw tensor object flows into `fwd_fn`.
- **Box**: wrap the raw output back into `MiniTensor` — that wrapper's `.array` is the SAME object as `out_raw`.

This invariant matters: cached-value reuse in backward fns (e.g. `sigmoid_back` reads `out` to compute `out*(1-out)`) depends on `is` identity. A `clone()` in either path silently breaks that and doubles memory.

### Composite Exercise — round-trip array ↔ MiniTensor with shared storage (no copy)

**Atoms exercised together**: `box-array-to-tensor-with-recipe`, `unbox-args-tensor-to-array`

Implement `cx10_round_trip(fwd_fn, args, kwargs)` — the two halves of `wrap_forward_fn` chained together, with NO COPYING anywhere:

1. Unbox: build `raw_args` by replacing each `MiniTensor` in `args` with its `.array` (non-Tensors pass through, order preserved).
2. Call: `raw_out = fwd_fn(*raw_args, **kwargs)`.
3. Box: build `out = MiniTensor(raw_out, requires_grad=True)` with a Recipe attached.
4. Return `out`.

Identity contracts (the test checks these explicitly):
- `out.array is raw_out` (boxing wraps; doesn't copy).
- For every MiniTensor `m` in `args`, the corresponding entry in `out.recipe.args` is `m.array` itself (not a copy).

In [ ]:
# Fill in the function below, then run this cell. The test asserts the composition is correct.

def cx10_round_trip(fwd_fn, args, kwargs):
    """Unbox → fwd_fn → box. Zero-copy: storage shared at both boundaries."""
    raise NotImplementedError

def _test_cx10():
    # --- single Tensor: log(x) ---
    x_raw = t.tensor([1.0, t.e, t.e * t.e])
    x = MiniTensor(x_raw)
    out = cx10_round_trip(t.log, (x,), {})
    assert isinstance(out, MiniTensor)
    assert t.allclose(out.array, t.tensor([0.0, 1.0, 2.0]), atol=1e-5)
    assert out.requires_grad is True
    assert out.recipe is not None
    # --- box-side identity: out.array IS the raw output ---
    # We can't compare directly (we never see raw_out), so we check that
    # out.array shares storage with a recomputed raw output via data_ptr.
    recomputed = t.log(x.array)
    assert out.array.shape == recomputed.shape
    # stronger: confirm boxing didn't clone the input either
    assert out.recipe.args[0] is x.array, 'unbox-side identity: recipe stores x.array itself'
    # --- mutation propagation proves storage sharing ---
    x_raw2 = t.tensor([5.0, 6.0])
    x2 = MiniTensor(x_raw2)
    out2 = cx10_round_trip(t.clone, (x2,), {})  # clone always makes a fresh tensor
    # But the RECIPE'S args[0] must still be the original input storage:
    assert out2.recipe.args[0] is x2.array, 'recipe.args holds the original raw tensor'
    assert out2.recipe.args[0].data_ptr() == x_raw2.data_ptr(), 'storage identity'
    # Mutating x2.array via x_raw2 must show up through out2.recipe.args[0]:
    x_raw2[0] = 999.0
    assert out2.recipe.args[0][0].item() == 999.0, 'recipe.args[0] shares storage with the input'
    # --- mixed args: multiply(x, 3.0) — scalar passes through ---
    x3 = MiniTensor(t.tensor([2.0, 4.0]))
    out3 = cx10_round_trip(t.multiply, (x3, 3.0), {})
    assert t.allclose(out3.array, t.tensor([6.0, 12.0]))
    assert out3.recipe.args == (x3.array, 3.0), 'unbox preserved float and order'
    assert out3.recipe.args[0] is x3.array, 'identity preserved through round-trip'
    # --- two MiniTensors ---
    a = MiniTensor(t.tensor([1.0, 2.0]))
    b = MiniTensor(t.tensor([10.0, 20.0]))
    out4 = cx10_round_trip(t.multiply, (a, b), {})
    assert t.allclose(out4.array, t.tensor([10.0, 40.0]))
    assert out4.recipe.args[0] is a.array
    assert out4.recipe.args[1] is b.array
    _dd_passed.add('cx10')

_test_cx10()

<details><summary>Show solution — cx10</summary>

```python
from dataclasses import dataclass, field
from typing import Any, Callable, Optional

grad_tracking_enabled = True

@dataclass
class Recipe:
    func: Optional[Callable] = None
    args: tuple = ()
    kwargs: dict = field(default_factory=dict)
    parents: dict = field(default_factory=dict)

class MiniTensor:
    def __init__(self, array, requires_grad: bool = False, recipe=None):
        self.array = array
        self.requires_grad = requires_grad
        self.recipe = recipe

def cx10_round_trip(fwd_fn, args, kwargs):
    raw_args = tuple(
        a.array if isinstance(a, MiniTensor) else a
        for a in args
    )
    raw_out = fwd_fn(*raw_args, **kwargs)
    out = MiniTensor(raw_out, requires_grad=True)
    out.recipe = Recipe(fwd_fn, raw_args, kwargs, {})
    return out
```

**Zero-copy boundaries.** The unbox step reads `a.array` (a Python-level attribute read — free). The box step calls `MiniTensor(raw_out, ...)` whose `__init__` just stashes `raw_out` on `self.array` (no clone). Both boundaries are O(1) and storage-sharing.

**Why we test data_ptr().** `is` checks Python object identity; `data_ptr()` checks storage identity. A clone would give the same Python object handle nowhere (so `is` fails) AND a different data_ptr (so the data_ptr check fails). Either alone catches a stray `.clone()`.
</details>

## Report completion

Run the cell below to send progress to Delta Drills. The beacon fires once and reports all 2 subtopics together.

In [ ]:
# === Delta Drills completion beacon (composite — fires for ALL atoms) ===
import urllib.request as _dd_req, json as _dd_json

_DD_REQUIRED = {'cx10'}

def report_completion():
    missing = _DD_REQUIRED - _dd_passed
    if missing:
        print(f"[Delta Drills] {sorted(missing)} not yet passing — fix the cell above, then re-run this one.")
        return
    if not DD_TOKEN:
        print('[Delta Drills] DD_TOKEN is empty — completion not reported.')
        return
    body = _dd_json.dumps({
        'exercise_title': f'composite-drill:{DD_PRIMARY_ATOM}:cx10',
        'subtopics': ["Backprop: Box array as Tensor + recipe", "Backprop: Unbox Tensor args to array"],
        'feedback': 'somewhat',
        'correct': True,
    }).encode('utf-8')
    req = _dd_req.Request(
        f'{DD_BACKEND_URL}/api/practice/arena-rating',
        data=body,
        headers={
            'Content-Type': 'application/json',
            'Authorization': f'Bearer {DD_TOKEN}',
        },
        method='POST',
    )
    try:
        with _dd_req.urlopen(req, timeout=5) as r:
            resp = _dd_json.loads(r.read())
        print(f'[Delta Drills] reported composite (atoms={DD_ATOM_IDS})')
        print(f'[Delta Drills] EWMA updated: {resp}')
    except Exception as e:
        print(f'[Delta Drills] beacon failed: {e}')

report_completion()